# Real-Time Delta Lake Profiling

**Objective**: Inspect the streaming data lake, verify Medallion layers, and demonstrate Delta 'Time Travel'.

---

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from delta import DeltaTable
from pyspark.sql import SparkSession

# Setup plotting
sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Initialize Spark for Delta Lake
We need to configure the Spark Session to understand the Delta format and transaction logs.

In [ ]:
spark = SparkSession.builder \
    .appName("DeltaLakeProfiling") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("✅ Spark Session Active with Delta Support")

## 2. Inspect the Medallion Layers
We query the **Bronze** (Raw) and **Silver** (Clean) tables to verify the transformation and quality gates.

In [ ]:
path_bronze = "/app/data/delta/bronze/iot_raw"
path_silver = "/app/data/delta/silver/iot_clean"
path_quarantine = "/app/data/delta/quarantine/iot_events_anomaly"

print("📊 Layer Statistics:")
if os.path.exists(path_bronze):
    df_bronze = spark.read.format("delta").load(path_bronze)
    print(f"  - Bronze (Raw): {df_bronze.count()} records")

if os.path.exists(path_silver):
    df_silver = spark.read.format("delta").load(path_silver)
    print(f"  - Silver (Clean): {df_silver.count()} records")

if os.path.exists(path_quarantine):
    df_quarantine = spark.read.format("delta").load(path_quarantine)
    print(f"  - Quarantine (Anomalies): {df_quarantine.count()} records")

## 3. Delta 'Time Travel' (History Tracking)
One of the key features of Delta Lake is the ability to audit changes and roll back to previous versions.

In [ ]:
if os.path.exists(path_silver):
    dt = DeltaTable.forPath(spark, path_silver)
    print("📜 Delta Table History (Last 5 transactions):")
    display(dt.history(5).toPandas()[['version', 'timestamp', 'operation', 'operationMetrics']])

## 4. Real-Time Distribution Analysis
We visualize the distribution of temperatures in the Silver layer vs the Raw layer to prove the filters work.

In [ ]:
if os.path.exists(path_silver):
    pdf = df_silver.toPandas()
    sns.histplot(pdf['value'], kde=True, color='blue', label='Silver (Clean)')
    plt.title("Post-Processing Value Distribution")
    plt.xlabel("Temperature (C)")
    plt.show()